[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/05_%E3%83%81%E3%83%A3%E3%83%8D%E3%83%AB%E5%88%A5%E7%8D%B2%E5%BE%97%E7%8E%87%E3%81%AE%E6%AF%94%E8%BC%83.ipynb)

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに data/ が無ければ公開リポジトリ(raw)から読み込む
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
print('準備OK')
from scipy import stats

# 発展マーケ-05. チャネル別獲得率の比較（分散分析）

> Colab（ブラウザ）で実行できます。**最初に「① セットアップ」セルを実行**してください。
> **統計検定2級レベル**（実務でよく使う検定・回帰）。scipy・statsmodels を使います（Colabは導入済み。ローカルは `uv add scipy statsmodels`）。

**舞台設定**：チャネルは5つ。「獲得率(CVR)はチャネルによって違うのか」を、**一度に**検定したい。t検定を総当たりすると偽陽性が増えるので、**分散分析(ANOVA)**を使います。

**使う統計（2級）**：一元配置分散分析・F検定・多重比較の注意。

### 使うデータ：`web_marketing.csv`（月×チャネルのマーケ実績30行）
1行＝ある月・あるチャネルの実績。`獲得数 / クリック数 = CVR(獲得率)` をチャネル間で比べます。

## この章でできるようになること
- 3グループ以上の平均の差を **一元配置分散分析** で一度に検定できる
- F値・p値から「どこかに差があるか」を判断できる
- t検定の総当たり（多重比較）の問題点を説明できる

> **前提**：2標本t検定（発展04）　/　**所要**：約25分　/　**レベル**：2級

## 1. 各チャネルのCVR（獲得率）を出す

In [ ]:
mk = load('web_marketing.csv')
mk['CVR'] = mk['獲得数'] / mk['クリック数']              # 獲得率＝獲得÷クリック
print(mk.groupby('チャネル')['CVR'].agg(['mean','std','count']).round(3))
fig, ax = plt.subplots(figsize=(7,4))
mk.boxplot(column='CVR', by='チャネル', ax=ax); plt.suptitle(''); ax.set_title('チャネル別CVR')
ax.set_ylabel('CVR'); plt.show()

## 2. なぜ t検定の総当たりはダメ？

5チャネルを2つずつt検定すると **10通り**。それぞれ5%の偽陽性を許すと、全体では「どれか1つが偶然有意」になる確率がぐっと上がります。そこで **分散分析(ANOVA)** で「どこかに差があるか」を1回で検定します。

- H₀：すべてのチャネルの母平均CVRは等しい
- H₁：少なくとも1つは異なる

In [ ]:
groups = [g['CVR'].values for _, g in mk.groupby('チャネル')]
F, p = stats.f_oneway(*groups)                          # 一元配置分散分析
print(f'F統計量 = {F:.2f},  p値 = {p:.6f}')
print('→ 有意水準5%で', ('少なくとも1チャネルは違う' if p < 0.05 else '差があるとは言えない'))

## 3. F値の正体：群間のばらつき ÷ 群内のばらつき

F = 「チャネル**間**の差の大きさ」÷「チャネル**内**のばらつき」。グループ間の差が、グループ内の偶然のゆらぎより十分大きいと F が大きくなり、有意になります。

> グループが2つだけのとき、ANOVA は2標本t検定と一致します（**F = t²**）。

## まとめ
- 3グループ以上の平均比較は **一元配置分散分析(ANOVA)** で一度に検定する。
- t検定の総当たりは **多重比較**で偽陽性が増えるので避ける。
- F値は「群間の差 ÷ 群内のばらつき」。2群なら F = t²。

> **よくある間違い**
> - ANOVAが有意でも「**どのチャネルが**違うか」までは分からない → 多重比較法（Tukey等）が必要。
> - 各群の分散が大きく違う・正規から外れる場合は、結果の解釈に注意（前提のチェック）。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
from scipy import stats
# Q1: [1,2,3],[2,3,4],[6,7,8] の一元配置分散分析のp値を ans に
ans = None   # 例: stats.f_oneway([1,2,3],[2,3,4],[6,7,8]).pvalue
_check('Q1 ANOVAのp値', ans, stats.f_oneway([1,2,3],[2,3,4],[6,7,8]).pvalue)

In [ ]:
# Q2: 2群の比較で t統計量が 3 のとき、対応するF値（F=t²）を ans に
ans = None   # 例: 3**2
_check('Q2 F=t²', ans, 3**2)

In [ ]:
# Q3: 5チャネルを2つずつt検定する組み合わせの数 C(5,2) を ans に
ans = None   # 例: 5*4//2
_check('Q3 ペア数', ans, 5*4//2)

---
## 練習問題

**問1.** CVRではなく **CTR（クリック数/表示回数）** でチャネル間のANOVAを実行しよう。

**問2.** 一番CVRが高いチャネルと低いチャネルだけを取り出して2標本t検定し、F=t² になるか確かめよう。

**問3.**（考察）ANOVAで「有意差あり」と出た後、現場で次にやるべき分析は何か（どのチャネルが違うかを知るには）？

> **解答例は別ページ**（ネタバレ防止）**[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/05_%E3%83%81%E3%83%A3%E3%83%8D%E3%83%AB%E5%88%A5%E7%8D%B2%E5%BE%97%E7%8E%87%E3%81%AE%E6%AF%94%E8%BC%83.md)**
>
> 自分で解いてから開きましょう。

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 一元配置分散分析 | 1要因で3群以上の平均を比較 |
| F統計量 | 群間のばらつき ÷ 群内のばらつき |
| 多重比較 | 何度も検定して偽陽性が増える問題 |
| F = t² | 2群ではANOVAとt検定が一致 |

```python
from scipy import stats
stats.f_oneway(g1, g2, g3, ...)   # 一元配置分散分析
```

## 完了ログ

このノートを終えたら下のセルを実行すると、学習の完了が記録されます。
**学習者コードは最初に一度だけ設定**すれば、以降は全ノートで自動送信されます（名前の再入力は不要）。

- Colab：左の鍵アイコン（シークレット）で `STUDENT_CODE` に配布コードを登録（1回だけ）
- ローカル：環境変数 `STUDENT_CODE` を設定（または初回に画面入力すると保存され、次回から不要）

In [ ]:
# === 完了ログ ===  学習者コードは最初に一度だけ設定すれば、全ノートで自動送信されます。
import os, datetime, pathlib
try:
    import requests
except ImportError:
    requests = None

def _student_code():
    try:                                          # Colab: 鍵アイコンで STUDENT_CODE を登録
        from google.colab import userdata
        c = userdata.get('STUDENT_CODE')
        if c: return c.strip()
    except Exception:
        pass
    c = os.environ.get('STUDENT_CODE')            # ローカル: 環境変数
    if c: return c.strip()
    p = pathlib.Path.home() / '.student_code'      # 保存ファイル
    if p.exists(): return p.read_text().strip()
    try:                                           # 最後の手段: 入力して保存（次回から不要）
        c = input('学習者コードを入力（配布されたもの）: ').strip()
        if c: p.write_text(c); return c
    except Exception:
        pass
    return ''

CODE = _student_code()
LOG_URL = ""      # 配布時に設定
LOG_TOKEN = ""    # 配布時に設定
NOTEBOOK = "06_発展_マーケ分析/05_チャネル別獲得率の比較"

if CODE and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "code": CODE, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {CODE} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
elif not CODE:
    print("学習者コード未設定。Colabは鍵アイコンで STUDENT_CODE を登録、ローカルは環境変数 STUDENT_CODE を設定してください。")
else:
    print(f"{NOTEBOOK}: LOG_URL/LOG_TOKEN が未設定です（配布時に設定されます）。")